# MineDB Unified Self-Bootstrapping Pipeline (`Qwen/Qwen2.5-14B-Instruct-AWQ` + MotherDuck)
Triggered automatically via GitHub Actions (`kaggle kernels push`).

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import asyncio
import json
import os
import re
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from kaggle_secrets import UserSecretsClient

# ----------------------------- CONFIG ---------------------------------------
DUCKDB_VERSION = "1.5.6"                  # must match MotherDuck-supported version
MODEL_ID = "Qwen/Qwen2.5-14B-Instruct-AWQ"
DATASET_SLUG = "minedb-cache-14b-awq"     # stored as <KAGGLE_USERNAME>/minedb-cache-14b-awq
DATASET_TITLE = "MineDB Cache 14B AWQ"

BUFFER_PATH = "/kaggle/input/raw-scrape-buffer/raw_batch.parquet"
CACHE_TMP_ROOT = "/tmp/minedb_cache"      # uses /tmp so /kaggle/working stays clean on save
MAX_PER_QUERY = 10
FETCH_CONCURRENCY = 15
RAW_TEXT_CHARS = 1500
LLM_BATCH_SIZE = 4                        # ~10GB AWQ weights leave ample headroom on T4 x2
MAX_NEW_TOKENS = 256
MIN_VALUE_SCORE = 7
RETENTION_DAYS = 60
SEARCH_DELAY_S = 1.5                      # avoid DDG rate limiting

NICHE_QUERIES = {
    "real_estate": [
        "UAE property price drop below market",
        "Dubai Abu Dhabi distressed villa apartment sale",
    ],
    "b2b_leads": [
        "hiring software engineer cloud devops UAE",
        "series A funding startup hiring remote",
    ],
    "ecommerce": [
        "site:myshopify.com new collection price",
        "electronics clearance sale stock UAE",
    ],
    "tenders": [
        "request for proposal RFP tender submission deadline 2026",
        "government procurement tender notice UAE",
    ],
    "arabic_nlp": [
        "منتدى نقاش تجارب اسعار الامارات",
        "اسئلة واجوبة تقنية اعمال الخليج",
    ],
}

NICHE_SCHEMA = {
    "real_estate": '{"price":..., "currency":..., "location":..., "property_type":..., "urgency_signal":...}',
    "b2b_leads":   '{"company_name":..., "job_title":..., "location":..., "tech_stack":[...], "contact_or_url":...}',
    "ecommerce":   '{"product_name":..., "brand":..., "price":..., "discount_pct":..., "stock_status":...}',
    "tenders":     '{"tender_title":..., "issuing_org":..., "submission_deadline":..., "sector":...}',
    "arabic_nlp":  '{"clean_arabic_text":..., "dialect":..., "domain_category":...}',
}

SYSTEM_PROMPT = (
    "You are a strict commercial data quality gatekeeper. Read the text and reply with ONE JSON object and "
    "nothing else, with exactly these keys: "
    '"is_sellable" (true ONLY if the text contains a concrete, specific commercial item such as an actual '
    "property listing with price/location, a specific open job role, a concrete product offer, or a specific "
    "tender notice; false for directory/aggregator homepages, generic blogs, login pages, navigation junk, or spam), "
    '"value_score" (integer 1-10), "extracted_data" (object strictly following the schema given by the user; '
    "use JSON null for unknown fields—never write strings like 'Unknown' or 'N/A', and never invent values)."
)

# ----------------------------- SECRETS & SELF-BOOTSTRAPPING CACHE -----------
secrets = UserSecretsClient()

def get_secret_safe(name, required=True):
    val = os.environ.get(name)
    if not val:
        try:
            val = secrets.get_secret(name)
        except Exception:
            val = None
    if val:
        return val.strip().strip('"').strip("'")
    if required:
        raise ValueError(f"Missing required Kaggle Secret: '{name}'. Enable it in Kaggle Add-ons -> Secrets.")
    return None

md_token = get_secret_safe("MOTHERDUCK_TOKEN", required=True)
if not md_token.startswith("eyJ"):
    raise ValueError(
        "MOTHERDUCK_TOKEN does not look like a JWT (expected to start with 'eyJ'). "
        "Check Kaggle Add-ons -> Secrets."
    )
os.environ["MOTHERDUCK_TOKEN"] = md_token

kaggle_user = get_secret_safe("KAGGLE_USERNAME", required=True)
os.environ["KAGGLE_USERNAME"] = kaggle_user

kaggle_api_token = get_secret_safe("KAGGLE_API_TOKEN", required=False)
if kaggle_api_token:
    os.environ["KAGGLE_API_TOKEN"] = kaggle_api_token
else:
    os.environ["KAGGLE_KEY"] = get_secret_safe("KAGGLE_KEY", required=True)

DATASET_ID = f"{kaggle_user}/{DATASET_SLUG}"

REQUIRED_PKGS = [
    f"duckdb=={DUCKDB_VERSION}",
    "httpx",
    "trafilatura",
    "ddgs",
    "accelerate",
    "autoawq",
]


def find_existing_cache():
    for root_dir in ("/kaggle/input", CACHE_TMP_ROOT):
        if not os.path.exists(root_dir):
            continue
        wheels_dir, model_dir = None, None
        for r, dirs, _ in os.walk(root_dir):
            if "wheels" in dirs and wheels_dir is None:
                wheels_dir = os.path.join(r, "wheels")
            if "model" in dirs and model_dir is None:
                model_dir = os.path.join(r, "model")
            if wheels_dir and model_dir:
                return wheels_dir, model_dir
    return None, None


def ensure_offline_cache():
    wheels_dir, model_dir = find_existing_cache()
    if wheels_dir and model_dir:
        print(f"Step 1/6: Found local cache -> wheels: {wheels_dir} | model: {model_dir}", flush=True)
        return wheels_dir, model_dir

    # Check if the private Kaggle Dataset already exists remotely; if so, pull it via Kaggle CLI
    status = subprocess.run(
        ["kaggle", "datasets", "status", DATASET_ID],
        capture_output=True,
        text=True,
    )
    if status.returncode == 0:
        print(f"Step 1/6: Pulling existing cache dataset '{DATASET_ID}' into {CACHE_TMP_ROOT}...", flush=True)
        shutil.rmtree(CACHE_TMP_ROOT, ignore_errors=True)
        os.makedirs(CACHE_TMP_ROOT, exist_ok=True)
        subprocess.run(
            ["kaggle", "datasets", "download", "-d", DATASET_ID, "-p", CACHE_TMP_ROOT, "--unzip", "-q"],
            check=True,
        )
        wheels_dir, model_dir = find_existing_cache()
        if wheels_dir and model_dir:
            return wheels_dir, model_dir

    # First-ever run: build wheels + 14B AWQ snapshot and publish as a private Kaggle Dataset
    print(f"Step 1/6: First-time bootstrap -> building cache & publishing '{DATASET_ID}'...", flush=True)
    shutil.rmtree(CACHE_TMP_ROOT, ignore_errors=True)
    wheels_dir = os.path.join(CACHE_TMP_ROOT, "wheels")
    model_dir = os.path.join(CACHE_TMP_ROOT, "model")
    os.makedirs(wheels_dir, exist_ok=True)
    os.makedirs(model_dir, exist_ok=True)

    print("  [Bootstrap 1/3] Downloading Python wheels...", flush=True)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "download", "-q", "--dest", wheels_dir, *REQUIRED_PKGS],
        timeout=300,
    )

    print(f"  [Bootstrap 2/3] Downloading {MODEL_ID} snapshot (~10GB)...", flush=True)
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=model_dir,
        ignore_patterns=["*.msgpack", "*.h5", "*.ot", "*.tflite"],
    )
    shutil.rmtree(os.path.join(model_dir, ".cache"), ignore_errors=True)

    print(f"  [Bootstrap 3/3] Publishing '{DATASET_ID}' to Kaggle Datasets for future runs...", flush=True)
    meta = {
        "title": DATASET_TITLE,
        "id": DATASET_ID,
        "licenses": [{"name": "CC0-1.0"}],
    }
    with open(os.path.join(CACHE_TMP_ROOT, "dataset-metadata.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    subprocess.run(
        ["kaggle", "datasets", "create", "-p", CACHE_TMP_ROOT, "--dir-mode", "zip"],
        check=True,
    )
    print(f"  Published '{DATASET_ID}' successfully!", flush=True)
    return wheels_dir, model_dir


WHEELS_DIR, CACHED_MODEL_DIR = ensure_offline_cache()

# Install dependencies strictly from the offline wheel cache (force-reinstall duckdb==1.5.6 over Kaggle's 1.3.2)
print("  Installing dependencies from offline wheel cache...", flush=True)
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "--no-index",
        f"--find-links={WHEELS_DIR}", "--force-reinstall", "--no-deps",
        f"duckdb=={DUCKDB_VERSION}",
    ],
    timeout=120,
)
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "--no-index",
        f"--find-links={WHEELS_DIR}",
        *[p for p in REQUIRED_PKGS if not p.startswith("duckdb==")],
    ],
    timeout=240,
)

import duckdb
import httpx
import pandas as pd
import torch
import trafilatura
from transformers import AutoModelForCausalLM, AutoTokenizer

try:
    from ddgs import DDGS
except ImportError:
    from duckduckgo_search import DDGS

print(f"  DuckDB ready: v{duckdb.__version__} ({duckdb.__file__})", flush=True)
if duckdb.__version__ != DUCKDB_VERSION:
    raise RuntimeError(
        f"DuckDB v{duckdb.__version__} was already loaded in memory; click 'Run -> Restart & Clear Cell Outputs' and re-run."
    )


def run_async(coro):
    """Works both in plain scripts and inside Kaggle/Jupyter (running event loop)."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    with ThreadPoolExecutor(max_workers=1) as ex:
        return ex.submit(asyncio.run, coro).result()


In [ ]:
print(f"Step 2/6: Connecting to MotherDuck 'minedb' (DuckDB v{duckdb.__version__})...", flush=True)
con = duckdb.connect("md:minedb")

con.execute("""
    CREATE TABLE IF NOT EXISTS seen_urls (
        source_url VARCHAR PRIMARY KEY,
        first_seen TIMESTAMPTZ
    )
""")
for _niche in NICHE_QUERIES:  # fixed keys only, so f-string table names are safe
    con.execute(f"""
        CREATE TABLE IF NOT EXISTS {_niche} (
            niche VARCHAR, source_url VARCHAR, scraped_at TIMESTAMPTZ,
            value_score INTEGER, cleaned_json VARCHAR
        )
    """)
print("Connected.", flush=True)


In [ ]:
# ----------------------------- DISCOVERY ------------------------------------
def discover_urls():
    found, seen = [], set()
    with DDGS() as ddgs:
        for niche, queries in NICHE_QUERIES.items():
            for q in queries:
                try:
                    for r in ddgs.text(q, timelimit="d", max_results=MAX_PER_QUERY) or []:
                        href = r.get("href")
                        if href and href not in seen:
                            seen.add(href)
                            found.append((niche, href))
                except Exception as e:
                    print(f"Search warning [{niche}]: {e}", flush=True)
                time.sleep(SEARCH_DELAY_S)
    return found


def filter_unseen(pairs):
    existing = {r[0] for r in con.execute("SELECT source_url FROM seen_urls").fetchall()}
    return [(n, u) for n, u in pairs if u not in existing]


# ----------------------------- SCRAPING -------------------------------------
async def fetch_and_extract(client, sem, niche, url):
    async with sem:
        try:
            resp = await client.get(url, timeout=15.0)
            resp.raise_for_status()
            if "html" not in resp.headers.get("content-type", "").lower():
                return None
            text = await asyncio.to_thread(
                trafilatura.extract, resp.text, include_comments=False, include_tables=True
            )
            if text and len(text) > 150:
                return {
                    "niche": niche,
                    "source_url": url,
                    "raw_text": text[:RAW_TEXT_CHARS],
                    "scraped_at": datetime.now(timezone.utc),
                }
        except Exception:
            pass
    return None


async def scrape(targets):
    sem = asyncio.Semaphore(FETCH_CONCURRENCY)
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
    limits = httpx.Limits(max_connections=FETCH_CONCURRENCY * 2)
    async with httpx.AsyncClient(headers=headers, follow_redirects=True, limits=limits) as client:
        results = await asyncio.gather(*(fetch_and_extract(client, sem, n, u) for n, u in targets))
    return pd.DataFrame([r for r in results if r])


print("Step 3/6: Collecting pages...", flush=True)
if os.path.exists(BUFFER_PATH):
    df = pd.read_parquet(BUFFER_PATH)
    df["scraped_at"] = pd.to_datetime(df["scraped_at"], utc=True)
    df = df.drop_duplicates("source_url")
    seen_set = {u for (u,) in con.execute("SELECT source_url FROM seen_urls").fetchall()}
    df = df[~df["source_url"].isin(seen_set)]
    attempted = df["source_url"].tolist()
    print(f"Loaded {len(df)} unseen rows from buffer.", flush=True)
else:
    targets = filter_unseen(discover_urls())
    attempted = [u for _, u in targets]
    print(f"Fetching {len(targets)} brand-new URLs...", flush=True)
    df = run_async(scrape(targets)) if targets else pd.DataFrame()


def mark_seen(urls):
    if not urls:
        return
    now = datetime.now(timezone.utc)
    seen_df = pd.DataFrame({"source_url": list(dict.fromkeys(urls)), "first_seen": now})
    con.register("seen_df_view", seen_df)
    con.execute(
        "INSERT OR IGNORE INTO seen_urls (source_url, first_seen) "
        "SELECT source_url, first_seen FROM seen_df_view"
    )
    con.unregister("seen_df_view")


def apply_retention():
    cutoff = f"CAST(NOW() AS TIMESTAMPTZ) - INTERVAL '{RETENTION_DAYS} days'"
    con.execute(f"DELETE FROM seen_urls WHERE first_seen < {cutoff}")
    for n in NICHE_QUERIES:
        con.execute(f"DELETE FROM {n} WHERE scraped_at < {cutoff}")


if df.empty:
    mark_seen(attempted)
    apply_retention()
    print("No new valid pages this cycle. Exiting cleanly.", flush=True)
    con.close()
    sys.exit(0)


In [ ]:
# ----------------------------- PII SCRUB (Arabic NLP set) -------------------
EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
PHONE_RE = re.compile(r"(?<!\d)(?:\+?\d[\d\s().-]{7,}\d)(?!\d)")

def scrub_pii(text: str) -> str:
    return PHONE_RE.sub("[PHONE]", EMAIL_RE.sub("[EMAIL]", text))

mask = df["niche"] == "arabic_nlp"
df.loc[mask, "raw_text"] = df.loc[mask, "raw_text"].map(scrub_pii)

# ----------------------------- LLM QUALITY GATE -----------------------------
print(f"Step 4/6: Loading {MODEL_ID} from {CACHED_MODEL_DIR} to filter {len(df)} pages...", flush=True)
tokenizer = AutoTokenizer.from_pretrained(CACHED_MODEL_DIR, padding_side="left", local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    CACHED_MODEL_DIR,
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=True,
)
model.eval()


def build_prompt(niche: str, text: str) -> str:
    user = f"Niche: {niche}\nSchema for extracted_data: {NICHE_SCHEMA[niche]}\n\nTEXT:\n{text}"
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


def parse_reply(reply: str):
    s, e = reply.find("{"), reply.rfind("}") + 1
    if s == -1 or e <= s:
        return None
    try:
        obj = json.loads(reply[s:e])
        if (
            obj.get("is_sellable") is True
            and int(obj.get("value_score", 0)) >= MIN_VALUE_SCORE
            and isinstance(obj.get("extracted_data"), dict)
            and obj["extracted_data"]
        ):
            return obj
    except Exception:
        pass
    return None


print("Step 5/6: Running batched quality gate...", flush=True)
df = df.assign(prompt=[build_prompt(n, t) for n, t in zip(df["niche"], df["raw_text"])])
df = df.assign(plen=df["prompt"].str.len()).sort_values("plen")
rows = list(df.itertuples())
sellable = []

for i in range(0, len(rows), LLM_BATCH_SIZE):
    batch = rows[i:i + LLM_BATCH_SIZE]
    inputs = tokenizer(
        [r.prompt for r in batch],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1536,
    ).to(model.device)
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = out[:, inputs["input_ids"].shape[1]:]
    for r, reply in zip(batch, tokenizer.batch_decode(new_tokens, skip_special_tokens=True)):
        obj = parse_reply(reply.strip())
        if obj:
            sellable.append({
                "niche": r.niche,
                "source_url": r.source_url,
                "scraped_at": r.scraped_at,
                "value_score": int(obj["value_score"]),
                "cleaned_json": json.dumps(obj["extracted_data"], ensure_ascii=False),
            })
    done = min(i + LLM_BATCH_SIZE, len(rows))
    print(f"  -> {done}/{len(rows)} processed | sellable so far: {len(sellable)}", flush=True)

print(f"Quality gate: {len(rows)} scraped -> {len(sellable)} kept ({len(rows) - len(sellable)} discarded).", flush=True)


In [ ]:
# ----------------------------- SAVE -----------------------------------------
print("Step 6/6: Saving sellable records & applying retention...", flush=True)
mark_seen(attempted)

if sellable:
    clean_df = pd.DataFrame(sellable).drop_duplicates(["niche", "source_url"])
    for niche, group in clean_df.groupby("niche"):
        con.register("group_view", group)
        con.execute(f"""
            INSERT INTO {niche} (niche, source_url, scraped_at, value_score, cleaned_json)
            SELECT niche, source_url, scraped_at, value_score, cleaned_json FROM group_view
        """)
        con.unregister("group_view")
        print(f"Saved {len(group)} rows into 'minedb.{niche}'.", flush=True)
    print("\n--- SAMPLE OUTPUT ---")
    print(clean_df[["niche", "value_score", "cleaned_json"]].head(5).to_string(index=False), flush=True)
else:
    print(f"No rows met value_score >= {MIN_VALUE_SCORE}. 'minedb' stays clean.", flush=True)

apply_retention()
con.close()
print("Run completed successfully.", flush=True)
